2.1 Define the Response Schema 

In [ ]:
from pydantic import BaseModel
from typing import List, Union, Optional


class TextBlock(BaseModel):
    type: str = "text"
    content: str


class ListBlock(BaseModel):
    type: str = "list"
    title: Optional[str]
    items: List[str]


class TableBlock(BaseModel):
    type: str = "table"
    title: Optional[str]
    columns: List[str]
    rows: List[List[str]]


class CompositeBlock(BaseModel):
    type: str = "composite"
    blocks: List[Union[TextBlock, ListBlock, TableBlock]]


ResponseBlock = Union[TextBlock, ListBlock, TableBlock, CompositeBlock]

2.2 Prompt That Forces Structured Output

In [ ]:
SYSTEM_PROMPT = """
You are a banking AI assistant.

You MUST respond only in valid JSON.
Do not include explanations outside JSON.
Use one of the following response types:
- text
- list
- table
- composite

Follow this schema strictly.
"""

2.3 Simulated LLM Call (Replace with OpenAI/Azure)

In [ ]:
import json

def call_llm(user_question: str) -> dict:
    """
    Simulated LLM response.
    In production, replace with Azure OpenAI / OpenAI API call.
    """

    if "documents" in user_question.lower():
        return {
            "type": "list",
            "title": "Documents Required for Home Loan",
            "items": [
                "Identity Proof (Aadhaar / PAN)",
                "Income Proof (Salary Slips or IT Returns)",
                "Bank Statements (6 months)",
                "Property Documents"
            ]
        }

    if "interest rate" in user_question.lower():
        return {
            "type": "table",
            "title": "Loan Interest Rates",
            "columns": ["Loan Type", "Rate", "Tenure"],
            "rows": [
                ["Home Loan", "8.5% - 9.2%", "Up to 30 years"],
                ["Personal Loan", "10.5% - 14%", "Up to 5 years"]
            ]
        }

    return {
        "type": "text",
        "content": "Please ask a loan-related question."
    }

2.4 FastAPI Endpoint (Validated Output)

In [ ]:
from fastapi import FastAPI, HTTPException
from pydantic import ValidationError

app = FastAPI()

@app.post("/chat")
def chat(question: str):
    try:
        llm_response = call_llm(question)

        # Validate response against schema
        validated = ResponseBlock.parse_obj(llm_response)

        return {
            "status": "success",
            "response": validated
        }

    except ValidationError as e:
        raise HTTPException(status_code=500, detail="Invalid LLM response format")

 Frontend Code (React)

In [ ]:
function RenderBlock({ block }) {
  switch (block.type) {
    case "text":
      return <p>{block.content}</p>;

    case "list":
      return (
        <div>
          {block.title && <h4>{block.title}</h4>}
          <ul>
            {block.items.map((item, idx) => (
              <li key={idx}>{item}</li>
            ))}
          </ul>
        </div>
      );

    case "table":
      return (
        <div>
          {block.title && <h4>{block.title}</h4>}
          <table border="1">
            <thead>
              <tr>
                {block.columns.map((col, idx) => (
                  <th key={idx}>{col}</th>
                ))}
              </tr>
            </thead>
            <tbody>
              {block.rows.map((row, rIdx) => (
                <tr key={rIdx}>
                  {row.map((cell, cIdx) => (
                    <td key={cIdx}>{cell}</td>
                  ))}
                </tr>
              ))}
            </tbody>
          </table>
        </div>
      );

    case "composite":
      return (
        <div>
          {block.blocks.map((b, idx) => (
            <RenderBlock key={idx} block={b} />
          ))}
        </div>
      );

    default:
      return <p>Unsupported response type</p>;
  }
}